# Graded Exercise — College Dataset (Linear Regression)

**Dataset:** Small version of College.xlsx — 20 universities, 18 features (name removed).

**Target:** `Grad.Rate` — graduation rate of the university.

**Instructions followed:**
- Remove university name column
- Descriptive stats rounded to 2 dp
- One-hot encode `Private` with `drop_first=True`
- 70:30 train/test split, `random_state=42`
- No scaling or normalisation
- Metrics: R², MSE, MAE on both train and test sets

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 1. Create Dataset

Small dataset mirroring College.xlsx structure:
- `Private` — Yes/No (categorical)
- Numeric features: Apps, Accept, Enroll, Top10perc, Top25perc, F.Undergrad, P.Undergrad, Outstate, Room.Board, Books, Personal, PhD, Terminal, S.F.Ratio, perc.alumni, Expend
- Target: `Grad.Rate`

In [3]:
data = {
    'Name':        ['Alpha Univ', 'Beta College', 'Gamma Inst', 'Delta Univ', 'Epsilon Col',
                    'Zeta Univ', 'Eta College', 'Theta Inst', 'Iota Univ', 'Kappa Col',
                    'Lambda Univ', 'Mu College', 'Nu Inst', 'Xi Univ', 'Omicron Col',
                    'Pi Univ', 'Rho College', 'Sigma Inst', 'Tau Univ', 'Upsilon Col'],
    'Private':     ['Yes','Yes','No','Yes','No','Yes','Yes','No','Yes','No',
                    'Yes','No','Yes','Yes','No','No','Yes','Yes','No','Yes'],
    'Apps':        [1660,2186,1428,417,193,587,353,1899,1267,494,
                    1420,3200,900,2100,650,1800,410,980,2500,760],
    'Accept':      [1232,1924,1097,349,146,479,340,1720,1080,430,
                    1100,2800,780,1900,580,1500,370,850,2100,640],
    'Enroll':      [721,512,336,137,55,158,103,489,306,128,
                    430,900,250,600,200,550,140,310,700,220],
    'Top10perc':   [23,16,22,60,16,38,17,15,25,28,
                    30,12,45,20,18,22,55,35,14,40],
    'Top25perc':   [52,29,50,89,44,62,45,41,63,56,
                    68,35,72,50,42,55,82,65,38,74],
    'F.Undergrad': [2885,2683,1036,510,249,678,416,1988,1145,490,
                    1800,3500,900,2400,750,2100,480,1200,3100,820],
    'P.Undergrad': [537,1227,99,63,869,41,230,1273,175,35,
                    200,800,50,300,400,600,30,150,900,80],
    'Outstate':    [7440,12280,11250,12960,7560,13500,10100,8750,14200,9800,
                    11000,6500,15000,10500,8200,9000,16000,12000,7000,13500],
    'Room.Board':  [3300,6450,3750,5450,4120,4600,3800,4200,5100,3600,
                    4000,3200,5500,4300,3900,4100,5800,4700,3500,5000],
    'Books':       [450,750,400,450,800,500,420,550,480,430,
                    460,380,520,470,410,490,560,510,370,530],
    'Personal':    [2200,1500,1165,875,1500,1200,1800,2000,1100,1400,
                    1300,1800,900,1500,1700,1600,1000,1200,2100,1100],
    'PhD':         [70,29,53,92,76,68,55,60,85,72,
                    78,45,90,65,58,70,88,80,50,83],
    'Terminal':    [78,30,66,97,72,75,63,68,91,80,
                    84,52,95,71,65,76,93,86,58,89],
    'S.F.Ratio':   [18.1,12.2,12.9,7.7,11.9,14.3,11.5,15.0,10.2,13.8,
                    12.0,18.5,8.5,13.5,16.0,14.0,9.0,11.0,17.5,10.5],
    'perc.alumni': [12,16,30,37,2,21,20,14,26,15,
                    22,8,35,18,10,20,40,28,6,32],
    'Expend':      [7041,10527,8735,19016,10922,11000,9500,8200,13500,9000,
                    10000,7500,16000,9800,8500,9200,18000,12000,7800,14000],
    'Grad.Rate':   [60,56,54,59,15,55,63,73,80,52,
                    65,45,82,58,48,62,88,72,40,78]
}

df = pd.DataFrame(data)
print(f"Shape: {df.shape}")
df.head()

Shape: (20, 19)


,Name,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Alpha Univ,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Beta College,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Gamma Inst,No,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Delta Univ,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Epsilon Col,No,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


## 2. Remove Name Column

University name is an identifier — not a predictor.

In [4]:
df = df.drop(columns=['Name'])
print(f"Shape after dropping Name: {df.shape}")
df.head()

Shape after dropping Name: (20, 18)


,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,No,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,No,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


## 3. Descriptive Statistics

`.describe()` on numeric columns, rounded to 2 decimal places.

In [5]:
df.describe().round(2)

,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
count,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.00
mean,1260.20,1070.85,362.25,27.55,55.60,1456.50,402.95,10827.00,4418.50,496.50,1447.00,68.35,74.45,12.90,20.60,11012.05,60.25
std,828.51,718.93,239.61,13.83,16.09,1015.94,407.48,2808.29,887.31,109.65,393.14,16.61,16.44,3.08,10.67,3433.94,16.67
min,193.00,146.00,55.00,12.00,29.00,249.00,30.00,6500.00,3200.00,370.00,875.00,29.00,30.00,7.70,2.00,7041.00,15.00
25%,563.75,466.75,153.50,16.75,43.50,636.00,75.75,8612.50,3787.50,427.50,1148.75,57.25,65.75,10.88,13.50,8676.25,53.50
50%,1123.50,965.00,308.00,22.50,53.50,1090.50,215.00,10750.00,4160.00,475.00,1450.00,70.00,75.50,12.55,20.00,9900.00,59.50
75%,1824.75,1555.00,521.50,35.75,65.75,2175.00,650.00,13095.00,5025.00,522.50,1725.00,80.75,86.75,14.48,28.50,12375.00,72.25
max,3200.00,2800.00,900.00,60.00,89.00,3500.00,1273.00,16000.00,6450.00,800.00,2200.00,92.00,97.00,18.50,40.00,19016.00,88.00


## 4. One-Hot Encode `Private`

`Private` is Yes/No — must be numeric for the model.

`drop_first=True` drops the redundant column (avoids dummy variable trap).

Result: `Private_Yes` = 1 if private, 0 if not.

In [6]:
df = pd.get_dummies(df, columns=['Private'], drop_first=True)
print(f"Columns after encoding: {df.columns.tolist()}")
df.head()

Columns after encoding: ['Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc', 'F.Undergrad', 'P.Undergrad', 'Outstate', 'Room.Board', 'Books', 'Personal', 'PhD', 'Terminal', 'S.F.Ratio', 'perc.alumni', 'Expend', 'Grad.Rate', 'Private_Yes']


,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Private_Yes
0,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60,True
1,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56,True
2,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54,False
3,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59,True
4,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15,False


## 5. Split Features and Target

In [7]:
X = df.drop(columns=['Grad.Rate'])
y = df['Grad.Rate']

print(f"Features (X): {X.shape}  →  {X.columns.tolist()}")
print(f"Target  (y): {y.shape}")

Features (X): (20, 17)  →  ['Apps', 'Accept', 'Enroll', 'Top10perc', 'Top25perc', 'F.Undergrad', 'P.Undergrad', 'Outstate', 'Room.Board', 'Books', 'Personal', 'PhD', 'Terminal', 'S.F.Ratio', 'perc.alumni', 'Expend', 'Private_Yes']
Target  (y): (20,)


## 6. Train/Test Split — 70:30, random_state=42

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Train: {X_train.shape[0]} rows")
print(f"Test : {X_test.shape[0]} rows")

Train: 14 rows
Test : 6 rows


## 7. Fit Model on Training Set

In [9]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


## 8. Model Coefficients (rounded to 2 dp)

In [10]:
coef_df = pd.DataFrame({
    'Feature':     ['Intercept'] + X.columns.tolist(),
    'Coefficient': np.round([model.intercept_] + list(model.coef_), 2)
})
coef_df

,Feature,Coefficient
0,Intercept,-587.58
1,Apps,0.15
2,Accept,-0.13
3,Enroll,0.76
4,Top10perc,-1.47
5,Top25perc,0.70
6,F.Undergrad,-0.21
7,P.Undergrad,-0.05
8,Outstate,0.01
9,Room.Board,0.00


## 9. Predictions on Train and Test Sets

In [12]:
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

print(f"Train predictions (first 5): {y_train_pred[:5].round(2)}")
print(f"Test  predictions (first 5): {y_test_pred[:5].round(2)}")

Train predictions (first 5): [45. 59. 40. 88. 58.]
Test  predictions (first 5): [ 181.27   61.11   99.59 -221.73  115.1 ]


## 10. Evaluation Metrics — R², MSE, MAE

Computed on both train and test sets, rounded to 2 decimal places.

In [13]:
metrics = pd.DataFrame({
    'Metric': ['R²', 'MSE', 'MAE'],
    'Train': [
        round(r2_score(y_train, y_train_pred), 2),
        round(mean_squared_error(y_train, y_train_pred), 2),
        round(mean_absolute_error(y_train, y_train_pred), 2)
    ],
    'Test': [
        round(r2_score(y_test, y_test_pred), 2),
        round(mean_squared_error(y_test, y_test_pred), 2),
        round(mean_absolute_error(y_test, y_test_pred), 2)
    ]
})
metrics

,Metric,Train,Test
0,R²,1.0,-197.98
1,MSE,0.0,16078.84
2,MAE,0.0,87.63


## 11. Interpretation

- **R²:** How much variance in Grad.Rate the model explains
- **MSE:** Average squared error (penalises large errors more)
- **MAE:** Average absolute error in graduation rate points
- If Train R² >> Test R² → overfitting — model memorised training data
- If Train ≈ Test → model generalises well

In [14]:
r2_train = round(r2_score(y_train, y_train_pred), 2)
r2_test  = round(r2_score(y_test, y_test_pred), 2)

print(f"Train R²: {r2_train}")
print(f"Test  R²: {r2_test}")
print()
if r2_train - r2_test > 0.2:
    print("⚠ Large gap between Train and Test R² → likely overfitting")
    print("  Small dataset with many features → model memorises rather than generalises")
else:
    print("✓ Train and Test R² are close → model generalises reasonably")

Train R²: 1.0
Test  R²: -197.98

⚠ Large gap between Train and Test R² → likely overfitting
  Small dataset with many features → model memorises rather than generalises
